# Synapse vs Databricks Data Validation

Reusable notebook that compares tables between Synapse and Databricks.
Define your table mappings below, then run all cells to get a full comparison report.

**Requirements:** `pyodbc`, `polars`, `databricks-sql-connector`

In [1]:
import sys
import os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), ''))
sys.path.insert(0, os.path.abspath('..'))

import pyodbc
from databricks import sql
import polars as pl

from validation_utils import compare_synapse_vs_databricks

## Configuration

Edit the table mappings and connection details below.

In [2]:
# ─── Table Mappings ─────────────────────────────────────────────────────────────
# Dict of source_system → list of (synapse_schema, synapse_table, dbx_catalog, dbx_schema, dbx_table)
# Synapse source: dal_v views (current data only, no OMD filters needed)

TABLE_MAPPINGS = {
    "IOC_MMRS": [
        ("dal_v", "vw_UDLMMRSIOC_mmrs_cs_vw_cycle", "rtio_tactical_sourcealigned", "ioc_mmrs", "mmrs_vw_cycle"),
        ("dal_v", "vw_UDLMMRSIOC_mmrs_cs_vw_event", "rtio_tactical_sourcealigned", "ioc_mmrs", "mmrs_vw_event"),
        ("dal_v", "vw_UDLMMRSIOC_vw_location", "rtio_tactical_sourcealigned", "ioc_mmrs", "mmrs_vw_location"),
        ("dal_v", "vw_UDLMMRSIOC_vw_rf_material", "rtio_tactical_sourcealigned", "ioc_mmrs", "mmrs_vw_rf_material"),
        ("dal_v", "vw_UDLMMRSIOC_vw_rf_timecode", "rtio_tactical_sourcealigned", "ioc_mmrs", "mmrs_vw_rf_timecode"),
        ("dal_v", "vw_UDLMMRSIOC_vw_rf_unit", "rtio_tactical_sourcealigned", "ioc_mmrs", "mmrs_vw_rf_unit"),
        ("dal_v", "vw_UDLMMRSIOC_vw_unit_fleet_fleettype", "rtio_tactical_sourcealigned", "ioc_mmrs", "mmrs_vw_unit_fleet_fleettype"),
    ],
    "GENSD03_SICOM": [
        ("dal_v", "VW_UDLGENSD03_DBO_HSP_SUIVIETATEQUIPEMENTROWNUMBER_VIEW", "_ogr_azr_syd", "gensd03_sicom", "dbo_hsp_suivietatequipementrownumber_view"),
        ("dal_v", "VW_UDLGENSD03_DBO_GEN_CODEIPT_DEF", "_ogr_azr_syd", "gensd03_sicom", "dbo_gen_codeipt_def"),
        ("dal_v", "VW_UDLGENSD03_DBO_HSP_SUIVIETATEQUIPEMENT", "_ogr_azr_syd", "gensd03_sicom", "dbo_hsp_suivietatequipement"),
        ("dal_v", "VW_UDLGENSD03_DBO_HSP_GROUPEEQUIPEMENT_DEF", "_ogr_azr_syd", "gensd03_sicom", "dbo_hsp_groupeequipement_def"),
        ("dal_v", "VW_UDLGENSD03_DBO_HSP_MODELEEQUIPEMENT_DEF", "_ogr_azr_syd", "gensd03_sicom", "dbo_hsp_modeleequipement_def"),
        ("dal_v", "VW_UDLGENSD03_DBO_HSP_EQUIPEMENT_DEF", "_ogr_azr_syd", "gensd03_sicom", "dbo_hsp_equipement_def"),
    ],
    "PI_SYSTEMS": [
        ("dal_v", "vw_udlpisystems_opsdata_mis_rtit_headgrade", "lakehouse", "analytics_and_enablement_staging_pi_systems", "opsdata_mis_rtit_headgrade"),
        ("dal_v", "vw_udlpisystems_opsdata_cs_mis_rtit_sandmetrics", "lakehouse", "analytics_and_enablement_staging_pi_systems", "opsdata_mis_rtit_sandmetrics"),
        ("dal_v", "vw_udlpisystems_opsdata_cs_pm_reporting_data", "lakehouse", "analytics_and_enablement_staging_pi_systems", "opsdata_pm_reporting_data"),
        ("dal_v", "vw_udlpisystems_opsdata_cs_pm_reporting_recovery_data", "lakehouse", "analytics_and_enablement_staging_pi_systems", "opsdata_pm_reporting_recovery_data"),
        ("dal_v", "vw_udlpisystems_pm_reporting_locations_hierarchy", "lakehouse", "analytics_and_enablement_staging_pi_systems", "globalpi_pm_reporting_locations_hierarchy"),
    ],
    "ODP": [
# dbx        ("hstg_v", "vw_hstg_UDLDWBDP_DAL_CS_FACT_HMETIMEUSAGEEVENT", "ogr_sourcealigned", "odp", "dal_fact_hmetimeusageevent"),
# dbx       ("hstg_v", "vw_hstg_UDLDWBDP_DAL_DIM_HMEASSET", "ogr_sourcealigned", "odp", "dal_dim_hmeasset"),
# syp       ("hstg_v", "vw_hstg_UDLDWBDP_DAL_DIM_HMETIMEUSAGEMODEL", "ogr_sourcealigned", "odp", "dal_dim_hmetimeusagemodel"),
# syp       ("hstg_v", "vw_hstg_UDLDWBDP_DAL_DIM_LOCATION", "ogr_sourcealigned", "odp", "dal_dim_location"),
# syp       ("hstg_v", "vw_hstg_UDLDWBDP_DAL_DIM_PAYLOADMANAGEMENT", "ogr_sourcealigned", "odp", "dal_dim_payloadmanagement"),
# dbx       ("hstg_v", "vw_hstg_UDLDWBDP_DAL_FACT_HMELOADHAULCYCLE", "ogr_sourcealigned", "odp", "dal_fact_hmeloadhaulcycle"),
        ("dal_v", "vw_UDLODP_DAL_DIM_FIXEDPLANTMETRIC", "ogr_sourcealigned", "odp", "dal_dim_fixedplantmetric"),
        ("dal_v", "vw_UDLODP_DAL_DIM_MATERIAL", "ogr_sourcealigned", "odp", "dal_dim_material"),
        ("dal_v", "vw_UDLODP_DAL_DIM_SHIFT", "ogr_sourcealigned", "odp", "dal_dim_shift"),
        ("dal_v", "vw_UDLODP_DAL_DIM_SITE", "ogr_sourcealigned", "odp", "dal_dim_site"),
        ("dal_v", "vw_UDLODP_DAL_DIM_TARGET", "ogr_sourcealigned", "odp", "dal_dim_target"),
        ("dal_v", "vw_UDLODP_DAL_FACT_FIXEDPLANTPRODUCTIONSHIFT", "ogr_sourcealigned", "odp", "dal_fact_fixedplantproductionshift"),
        ("dal_v", "vw_UDLODP_DAL_FACT_FIXEDPLANTTARGET", "ogr_sourcealigned", "odp", "dal_fact_fixedplanttarget"),
    ],
    "EGRP_GREPORTS": [
        ("dal_v", "vw_UDLRIOTINTOSYDNEYINTERNALSAPB4P1B4PSAPB4P_B4P_G42413A_ZBB_QUALITY_LOCAL_GI_GOVERNED", "ent_sourcealigned", "egrp_greports", "g42413a_zbb_quality_local_gi_governed"),
        ("dal_v", "vw_UDLRIOTINTOSYDNEYINTERNALSAPB4P1B4PSAPB4P_B4P_G44561A_SCHEDULE_COMPLIANCE_GI_GOVERNED", "ent_sourcealigned", "egrp_greports", "g44561a_schedule_compliance_gi_governed"),
        ("dal_v", "vw_UDLRIOTINTOSYDNEYINTERNALSAPB4P1B4PSAPB4P_B4P_G44561A_SCHEDULE_WORK_GI_GOVERNED", "ent_sourcealigned", "egrp_greports", "g44561a_schedule_work_gi_governed"),
        ("dal_v", "vw_UDLRIOTINTOSYDNEYINTERNALSAPB4P1B4PSAPB4P_B4P_G44564A_SNAPSHOT_BACKLOG_AGE_GOVERNED", "ent_sourcealigned", "egrp_greports", "g44564a_snapshot_backlog_age_governed"),
        ("dal_v", "vw_UDLRIOTINTOSYDNEYINTERNALSAPB4P1B4PSAPB4P_B4P_G44565A_USE_OF_RESOURCE_AVAILABILITY_GI_GOVERNED", "ent_sourcealigned", "egrp_greports", "g44565a_use_of_resource_availability_gi_governed"),
        ("dal_v", "vw_UDLRIOTINTOSYDNEYINTERNALSAPB4P1B4PSAPB4P_B4P_G44567A_PRIMARY_MAINTENANCE_CALL_COMPLIANCE_GOVERNED", "ent_sourcealigned", "egrp_greports", "g44567a_primary_maintenance_call_compliance_governed"),
        ("dal_v", "vw_UDLRIOTINTOSYDNEYINTERNALSAPB4P1B4PSAPB4P_B4P_G44569A_MAINTENANCE_PLANS_FUNCTIONAL_LOCATION_CRI_LOOKUP", "ent_sourcealigned", "egrp_greports", "g44569a_maintenance_plans_functional_location_cri_lookup"),
        ("dal_v", "vw_UDLRIOTINTOSYDNEYINTERNALSAPB4P1B4PSAPB4P_B4P_G44576A_WORK_EVENT_PROFILE_GOVERNED", "ent_sourcealigned", "egrp_greports", "g44576a_work_event_profile_governed"),
        ("dal_v", "vw_UDLRIOTINTOSYDNEYINTERNALSAPB4P1B4PSAPB4P_B4P_G52002A_PRODUCT_QUALITY", "ent_sourcealigned", "egrp_greports", "g52002a_product_quality"),
        ("dal_v", "vw_UDLRIOTINTOSYDNEYINTERNALSAPB4P1B4PSAPB4P_B4P_G52004A_TOTAL_MATERIAL_MOVED_ACTUAL", "ent_sourcealigned", "egrp_greports", "g52004a_total_material_moved_actual"),
        ("dal_v", "vw_UDLRIOTINTOSYDNEYINTERNALSAPB4P1B4PSAPB4P_B4P_G70301A_MAT_MVMT_ANLYS_PRODFACTSHEET_GOV", "ent_sourcealigned", "egrp_greports", "g70301a_mat_mvmt_anlys_prodfactsheet_gov"),
    ],
    "EGRP_IAP_STAGING": [
        ("dal_v", "vw_UDLANALYTICSANDENABLEMENTREPOSITORYAZUREIAPSTAGING_EGR_MP_RDL_DIM_OPERATING_RESPONSIBILITY", "lakehouse", "analytics_and_enablement_repository_azure_iap_staging", "egr_mp_rdl_dim_operating_responsibility"),
        ("dal_v", "vw_UDLANALYTICSANDENABLEMENTREPOSITORYAZUREIAPSTAGING_EGR_MPA_ODL_DIM_FUNCTIONAL_LOCATION", "lakehouse", "analytics_and_enablement_repository_azure_iap_staging", "egr_mpa_odl_dim_functional_location"),
# syp        ("hstg_v", "vw_hstg_UDLANALYTICSANDENABLEMENTREPOSITORYAZUREIAPSTAGING_EGR_MMP_RDL_DIM_PROCESS_HIER", "lakehouse", "analytics_and_enablement_repository_azure_iap_staging", "egr_mmp_rdl_dim_process_hier"),
# syp       ("hstg_v", "vw_hstg_UDLANALYTICSANDENABLEMENTREPOSITORYAZUREIAPSTAGING_EGR_MP_RDL_DIM_OPERATING_RESPONSIBILITY_HIER", "lakehouse", "analytics_and_enablement_repository_azure_iap_staging", "egr_mp_rdl_dim_operating_responsibility_hier"),
# syp       ("hstg_v", "vw_hstg_UDLANALYTICSANDENABLEMENTREPOSITORYAZUREIAPSTAGING_EGR_MPA_RDL_DIM_ASSET", "lakehouse", "analytics_and_enablement_repository_azure_iap_staging", "egr_mpa_rdl_dim_asset"),
    ],
    "EGRP_SAP_BRONZE": [
        ("dal_v", "vw_UDLLIBECPRTP_SAP_CS_QMFE", "ent_sourcealigned", "egrp_sap_bronze", "qmfe"),
        ("dal_v", "vw_UDLLIBECPRTP_SAP_MAKT", "lakehouse", "lib_ecp_rtp", "sap_makt"),
    ],
}

In [3]:
# ─── Connection Setup ──────────────────────────────────────────────────────────

# Synapse
synapse_server = 'syn-rt-prd-unity.sql.azuresynapse.net'
synapse_database = 'syndwrtprdunity'
synapse_username = 'byambadorjme@riotinto.com'
synapse_driver = '{ODBC Driver 17 for SQL Server}'

synapse_connection_string = f"""
    DRIVER={synapse_driver};
    SERVER={synapse_server};
    DATABASE={synapse_database};
    UID={synapse_username};
    Authentication=ActiveDirectoryInteractive;
    Encrypt=yes;
    TrustServerCertificate=no;
    Connection Timeout=30;
"""

# Databricks
databricks_access_token = os.environ["DATABRICKS_TOKEN"]
databricks_server_hostname = 'adb-2439498039309203.3.azuredatabricks.net'
databricks_http_path = '/sql/1.0/warehouses/e7d134c4348ac8b5'

# ─── Connect ──────────────────────────────────────────────────────────────────
print("Connecting to Synapse (browser auth prompt may appear)...")
synapse_conn = pyodbc.connect(synapse_connection_string)
print("✓ Synapse connected")

databricks_conn = sql.connect(
    server_hostname=databricks_server_hostname,
    http_path=databricks_http_path,
    access_token=databricks_access_token,
)
print("✓ Databricks connected")

Connecting to Synapse (browser auth prompt may appear)...
✓ Synapse connected
✓ Databricks connected


In [5]:
# ─── Display settings ─────────────────────────────────────────────────────────
pl.Config.set_tbl_rows(100)
pl.Config.set_tbl_cols(50)
pl.Config.set_fmt_str_lengths(100)

polars.config.Config

## Run Comparison

Iterates over all table mappings and produces a summary report.

In [6]:
def run_comparison(source_system: str, mappings: list, synapse_conn, databricks_conn) -> list:
    """Run comparisons for a list of table mappings and return results."""
    print(f"\n{'█' * 80}")
    print(f"  SOURCE SYSTEM: {source_system} ({len(mappings)} tables)")
    print(f"{'█' * 80}")

    results = []
    for i, (syn_schema, syn_table, dbx_catalog, dbx_schema, dbx_table) in enumerate(mappings, 1):
        print(f"\n  [{i}/{len(mappings)}] {dbx_catalog}.{dbx_schema}.{dbx_table}")
        try:
            result = compare_synapse_vs_databricks(
                databricks_conn=databricks_conn, synapse_conn=synapse_conn,
                synapse_schema_name=syn_schema, synapse_table_name=syn_table,
                databricks_catalog_name=dbx_catalog, databricks_schema_name=dbx_schema,
                databricks_table_name=dbx_table,
            )
            result["source_system"] = source_system
            result["synapse_table"] = syn_table
            result["databricks_table"] = dbx_table
            results.append(result)

            match_icon = "✓" if result["row_count_matches"] else "✗"
            print(f"    Schema: {'✓' if result['schema_matches'] else '✗'} | "
                  f"Rows: {match_icon} Syn={result['synapse_row_count']:,} Dbx={result['databricks_row_count']:,} "
                  f"(diff={result['row_diff']:+,}, {result['row_diff_pct']:+.1f}%)")
        except Exception as e:
            print(f"    ✗ ERROR: {e}")
            results.append({"source_system": source_system, "synapse_table": syn_table,
                            "databricks_table": dbx_table, "error": str(e)})

    print(f"\n  Done: {len(results)} tables compared")
    return results

In [ ]:
results_ioc_mmrs = run_comparison("IOC_MMRS", TABLE_MAPPINGS["IOC_MMRS"], synapse_conn, databricks_conn)


████████████████████████████████████████████████████████████████████████████████
  SOURCE SYSTEM: IOC_MMRS (7 tables)
████████████████████████████████████████████████████████████████████████████████

  [1/7] rtio_tactical_sourcealigned.ioc_mmrs.mmrs_vw_cycle
    Schema: ✗ | Rows: ✗ Syn=5,712,266 Dbx=6,257 (diff=-5,706,009, -99.9%)

  [2/7] rtio_tactical_sourcealigned.ioc_mmrs.mmrs_vw_event
    Schema: ✓ | Rows: ✗ Syn=32,671,826 Dbx=56,110,770 (diff=+23,438,944, +71.7%)

  [3/7] rtio_tactical_sourcealigned.ioc_mmrs.mmrs_vw_location
    Schema: ✓ | Rows: ✗ Syn=12,727 Dbx=12,729 (diff=+2, +0.0%)

  [4/7] rtio_tactical_sourcealigned.ioc_mmrs.mmrs_vw_rf_material
    Schema: ✓ | Rows: ✓ Syn=40 Dbx=40 (diff=+0, +0.0%)

  [5/7] rtio_tactical_sourcealigned.ioc_mmrs.mmrs_vw_rf_timecode
    Schema: ✓ | Rows: ✓ Syn=986 Dbx=986 (diff=+0, +0.0%)

  [6/7] rtio_tactical_sourcealigned.ioc_mmrs.mmrs_vw_rf_unit
    Schema: ✓ | Rows: ✓ Syn=511 Dbx=511 (diff=+0, +0.0%)

  [7/7] rtio_tactical_sourcealigned

In [16]:
results_pi_system = run_comparison("PI_SYSTEMS", TABLE_MAPPINGS["PI_SYSTEMS"], synapse_conn, databricks_conn)


████████████████████████████████████████████████████████████████████████████████
  SOURCE SYSTEM: PI_SYSTEMS (5 tables)
████████████████████████████████████████████████████████████████████████████████

  [1/5] lakehouse.analytics_and_enablement_staging_pi_systems.opsdata_mis_rtit_headgrade
    Schema: ✓ | Rows: ✓ Syn=12,570 Dbx=12,570 (diff=+0, +0.0%)

  [2/5] lakehouse.analytics_and_enablement_staging_pi_systems.opsdata_mis_rtit_sandmetrics
    Schema: ✓ | Rows: ✗ Syn=87,277 Dbx=86,547 (diff=-730, -0.8%)

  [3/5] lakehouse.analytics_and_enablement_staging_pi_systems.opsdata_pm_reporting_data
    Schema: ✓ | Rows: ✗ Syn=151,406 Dbx=164,368 (diff=+12,962, +8.6%)

  [4/5] lakehouse.analytics_and_enablement_staging_pi_systems.opsdata_pm_reporting_recovery_data
    Schema: ✓ | Rows: ✗ Syn=2,492 Dbx=2,647 (diff=+155, +6.2%)

  [5/5] lakehouse.analytics_and_enablement_staging_pi_systems.globalpi_pm_reporting_locations_hierarchy
    Schema: ✓ | Rows: ✓ Syn=3,318 Dbx=3,318 (diff=+0, +0.0%)

 

In [17]:
results_odp = run_comparison("ODP", TABLE_MAPPINGS["ODP"], synapse_conn, databricks_conn)


████████████████████████████████████████████████████████████████████████████████
  SOURCE SYSTEM: ODP (7 tables)
████████████████████████████████████████████████████████████████████████████████

  [1/7] ogr_sourcealigned.odp.dal_dim_fixedplantmetric
    Schema: ✗ | Rows: ✓ Syn=139 Dbx=139 (diff=+0, +0.0%)

  [2/7] ogr_sourcealigned.odp.dal_dim_material
    Schema: ✗ | Rows: ✓ Syn=38 Dbx=38 (diff=+0, +0.0%)

  [3/7] ogr_sourcealigned.odp.dal_dim_shift
    Schema: ✗ | Rows: ✓ Syn=7 Dbx=7 (diff=+0, +0.0%)

  [4/7] ogr_sourcealigned.odp.dal_dim_site
    Schema: ✗ | Rows: ✓ Syn=10 Dbx=10 (diff=+0, +0.0%)

  [5/7] ogr_sourcealigned.odp.dal_dim_target
    Schema: ✗ | Rows: ✓ Syn=21 Dbx=21 (diff=+0, +0.0%)

  [6/7] ogr_sourcealigned.odp.dal_fact_fixedplantproductionshift
    Schema: ✗ | Rows: ✓ Syn=364,896 Dbx=364,896 (diff=+0, +0.0%)

  [7/7] ogr_sourcealigned.odp.dal_fact_fixedplanttarget
    Schema: ✗ | Rows: ✓ Syn=3,068,983 Dbx=3,068,983 (diff=+0, +0.0%)

  Done: 7 tables compared


In [33]:
results_odp[2]['schema_mismatches_df']

column_name,synapse_data_type,column_name_normalized,column_name_right,databricks_data_type,synapse_expected_type,matched_column
str,str,str,str,str,str,str
null,null,null,"""DATALINEAGE_SK""","""LONG""",null,"""DATALINEAGE_SK"""
"""EXECUTIONID""","""BIGINT""","""EXECUTIONID""","""EXECUTIONID""","""LONG""",null,"""EXECUTIONID"""
"""ISCURRENT""","""INT""","""ISCURRENT""","""ISCURRENT""","""BOOLEAN""","""BIT""","""ISCURRENT"""
"""ISSTUB""","""INT""","""ISSTUB""","""ISSTUB""","""BOOLEAN""","""BIT""","""ISSTUB"""
null,null,null,"""SHIFT_SK""","""LONG""",null,"""SHIFT_SK"""
"""SHIFTFINISHTIME""","""TIME""","""SHIFTFINISHTIME""","""SHIFTFINISHTIME""","""TIMESTAMP""","""DATETIME2""","""SHIFTFINISHTIME"""
"""SHIFTSTARTTIME""","""TIME""","""SHIFTSTARTTIME""","""SHIFTSTARTTIME""","""TIMESTAMP""","""DATETIME2""","""SHIFTSTARTTIME"""


In [18]:
results_egrp_greports = run_comparison("EGRP_GREPORTS", TABLE_MAPPINGS["EGRP_GREPORTS"], synapse_conn, databricks_conn)


████████████████████████████████████████████████████████████████████████████████
  SOURCE SYSTEM: EGRP_GREPORTS (11 tables)
████████████████████████████████████████████████████████████████████████████████

  [1/11] ent_sourcealigned.egrp_greports.g42413a_zbb_quality_local_gi_governed
    Schema: ✓ | Rows: ✓ Syn=189,850 Dbx=189,850 (diff=+0, +0.0%)

  [2/11] ent_sourcealigned.egrp_greports.g44561a_schedule_compliance_gi_governed
    Schema: ✓ | Rows: ✗ Syn=3,214,231 Dbx=3,205,656 (diff=-8,575, -0.3%)

  [3/11] ent_sourcealigned.egrp_greports.g44561a_schedule_work_gi_governed
    Schema: ✓ | Rows: ✓ Syn=2,289,763 Dbx=2,289,763 (diff=+0, +0.0%)

  [4/11] ent_sourcealigned.egrp_greports.g44564a_snapshot_backlog_age_governed
    Schema: ✓ | Rows: ✗ Syn=2,789,860 Dbx=2,790,134 (diff=+274, +0.0%)

  [5/11] ent_sourcealigned.egrp_greports.g44565a_use_of_resource_availability_gi_governed
    Schema: ✓ | Rows: ✓ Syn=205,622 Dbx=205,622 (diff=+0, +0.0%)

  [6/11] ent_sourcealigned.egrp_greports.

In [19]:
results_egrp_iap_staging = run_comparison("EGRP_IAP_STAGING", TABLE_MAPPINGS["EGRP_IAP_STAGING"], synapse_conn, databricks_conn)


████████████████████████████████████████████████████████████████████████████████
  SOURCE SYSTEM: EGRP_IAP_STAGING (2 tables)
████████████████████████████████████████████████████████████████████████████████

  [1/2] lakehouse.analytics_and_enablement_repository_azure_iap_staging.egr_mp_rdl_dim_operating_responsibility
    Schema: ✗ | Rows: ✗ Syn=41,037 Dbx=41,008 (diff=-29, -0.1%)

  [2/2] lakehouse.analytics_and_enablement_repository_azure_iap_staging.egr_mpa_odl_dim_functional_location
    Schema: ✗ | Rows: ✗ Syn=3,601,585 Dbx=3,538,179 (diff=-63,406, -1.8%)

  Done: 2 tables compared


In [20]:
results_egrp_sap_bronze = run_comparison("EGRP_SAP_BRONZE", TABLE_MAPPINGS["EGRP_SAP_BRONZE"], synapse_conn, databricks_conn)


████████████████████████████████████████████████████████████████████████████████
  SOURCE SYSTEM: EGRP_SAP_BRONZE (2 tables)
████████████████████████████████████████████████████████████████████████████████

  [1/2] ent_sourcealigned.egrp_sap_bronze.qmfe
    Schema: ✗ | Rows: ✗ Syn=8,450,783 Dbx=9,394,651 (diff=+943,868, +11.2%)

  [2/2] lakehouse.lib_ecp_rtp.sap_makt
    Schema: ✓ | Rows: ✗ Syn=21,175,004 Dbx=21,175,777 (diff=+773, +0.0%)

  Done: 2 tables compared


In [28]:
results_egrp_sap_bronze[0]['stats_comparison_df']

column,type,synapse_nulls,synapse_distinct,synapse_min,synapse_max,synapse_len_min,synapse_len_max,databricks_nulls,databricks_distinct,databricks_min,databricks_max,databricks_len_min,databricks_len_max
str,str,i64,i64,str,str,i64,i64,i64,i64,str,str,i64,i64
"""AEDAT_SIMP_DT""","""DATETIME2""",8077566,7235,"""2006-08-08 00:00:00""","""2026-07-22 00:00:00""",null,null,8980465,7244,"""2006-08-08""","""2026-07-22""",null,null
"""EKORG""","""NVARCHAR""",0,1,null,null,0,0,0,1,null,null,1,1
"""ERNAM""","""NVARCHAR""",0,25982,null,null,3,12,0,26368,null,null,3,12
"""ERZEIT""","""NVARCHAR""",0,86400,null,null,6,6,0,86400,null,null,6,6
"""FENUM""","""NVARCHAR""",0,19,null,null,4,4,0,19,null,null,4,4
"""FENUMORG""","""NVARCHAR""",0,1,null,null,4,4,0,1,null,null,4,4
"""KZLOESCH""","""NVARCHAR""",0,2,null,null,0,1,0,2,null,null,1,1
"""KZORG""","""NVARCHAR""",0,1,null,null,0,0,0,1,null,null,1,1
"""MATNR""","""NVARCHAR""",0,1,null,null,0,0,0,1,null,null,1,1


In [21]:
import pickle
from datetime import datetime

# ─── Save all results to disc ─────────────────────────────────────────────────
all_results = {
    "IOC_MMRS": results_ioc_mmrs,
    "PI_SYSTEMS": results_pi_system,
    "ODP": results_odp,
    "EGRP_GREPORTS": results_egrp_greports,
    "EGRP_IAP_STAGING": results_egrp_iap_staging,
    "EGRP_SAP_BRONZE": results_egrp_sap_bronze,
}

output_path = f"../output/validation_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pkl"
os.makedirs("../output", exist_ok=True)
with open(output_path, "wb") as f:
    pickle.dump(all_results, f)

print(f"✓ Saved {sum(len(v) for v in all_results.values())} results to {output_path}")
print(f"  Load with: all_results = pickle.load(open('{output_path}', 'rb'))")

✓ Saved 34 results to ../output/validation_results_20260722_170214.pkl
  Load with: all_results = pickle.load(open('../output/validation_results_20260722_170214.pkl', 'rb'))


In [ ]:
import pickle
all_results = pickle.load(open("../output/validation_results_20260722_HHMMSS.pkl", "rb"))

In [ ]:
# ─── Summary Report ───────────────────────────────────────────────────────────
summary_rows = []
for source_system, results in all_results.items():
    for r in results:
        if "error" in r:
            summary_rows.append({
                "source_system": source_system,
                "table": r["databricks_table"],
                "synapse_rows": None,
                "databricks_rows": None,
                "row_diff": None,
                "row_diff_pct": None,
                "schema_match": None,
                "row_count_match": None,
                "common_cols": None,
                "status": f"ERROR: {r['error'][:60]}",
            })
        else:
            status = "✓ Match" if r["row_count_matches"] and r["schema_matches"] else (
                "⚠ Row count mismatch" if not r["row_count_matches"] else "⚠ Schema mismatch"
            )
            summary_rows.append({
                "source_system": source_system,
                "table": r["databricks_table"],
                "synapse_rows": r["synapse_row_count"],
                "databricks_rows": r["databricks_row_count"],
                "row_diff": r["row_diff"],
                "row_diff_pct": r["row_diff_pct"],
                "schema_match": r["schema_matches"],
                "row_count_match": r["row_count_matches"],
                "common_cols": len(r["common_columns"]),
                "status": status,
            })

summary_df = pl.DataFrame(summary_rows)
print("\n" + "=" * 80)
print("VALIDATION SUMMARY")
print("=" * 80)
print(summary_df)

In [ ]:
# ─── Detailed Results (per table) ─────────────────────────────────────────────
# Uncomment a table index to inspect its detailed results

# idx = 0  # Change index to inspect a specific table
# r = all_results['GENSD03_SICOM'][idx]
# print(f"Table: {r['databricks_table']}")
# print(f"\nSchema Comparison:")
# display(r['schema_comparison_df'])
# print(f"\nColumn Stats Comparison:")
# display(r['stats_comparison_df'])

In [ ]:
# ─── Cleanup ──────────────────────────────────────────────────────────────────
synapse_conn.close()
databricks_conn.close()
print("Connections closed.")